# Attention Playground — coding workshop (beginner)

**Expected time:** ~15 minutes, individual, no GPU needed.

You just watched a tiny, hand-set example of attention: vectors with three dimensions, three keys, three queries, and 79% of the attention on the key after *tía* for “aunt”. Here you rebuild that exact example yourself, then go further: pull the real attention weights out of an actual trained translation model, on this exact Galician fragment, and see whether a real model is anywhere near as tidy as the toy one.


## How this workshop works

This is the coding version of the notebook, **beginner level**. Some code cells have gaps
marked `___`. An exercise above each such cell says what to write. This version takes
longer than the expected time given above.

1. Read the exercise. Open the hints if you need them.
2. Replace every `___` in the cell with your code, then run the cell.
3. Run the check cell right below it. It prints `✓ Exercise N passed`, or an error
   message that says what to look at.
4. If you are stuck, the solution is below the check cell. Open it after you have tried.

A cell that still contains `___` stops with `NameError: name '___' is not defined`.
Later cells use your code, so finish each exercise before moving on.

## Part A — Rebuild the toy attention example

Three keys and three queries, hand-set so the arithmetic comes out clean. This is the exact example from the lesson screen "Worked example: aligning “unha tía miña” → “an aunt of mine”" and from section 1 of the attention page — same numbers, same softmax.


### ✏️ Exercise 1 — Softmax and attention weights

Complete `softmax` and `attention_weights`.

- `softmax` turns any list of scores into weights that are positive and sum to 1.
- `attention_weights` scores each key against the query with a dot product, then applies
  softmax.

The cell should print the lecture's weights: 0.79 on the key after tía for “aunt”, 0.75 on the
key after unha for “an”, 0.91 on the key after miña for “mine”.

<details><summary><b>Hints</b> (click to open)</summary>

- softmax(s)_i = exp(s_i) / sum_j exp(s_j).
- `keys` has one key per row, so `keys @ query` gives one score per key.

</details>

In [ ]:
import numpy as np

word_vectors = {"unha": np.array([1.0, 0, 0]), "tía": np.array([0, 1.0, 0]), "miña": np.array([0, 0, 1.0])}

encoder_vectors = []
v = np.zeros(3)
for word in ["unha", "tía", "miña"]:
    v = 0.5 * v + 0.5 * word_vectors[word]
    encoder_vectors.append(v)
keys = np.array(encoder_vectors)
print("keys:", keys.tolist())


def softmax(scores):
    exp_scores = np.exp(scores - np.max(scores))
    return ___ / ___


def attention_weights(query, keys):
    scores = keys @ ___
    return ___(scores)


q_aunt = np.array([0, 6, 0])
weights_aunt = attention_weights(q_aunt, keys)
print(f"Writing 'aunt': after unha={weights_aunt[0]:.2f}, after tía={weights_aunt[1]:.2f}, after miña={weights_aunt[2]:.2f}  (lecture: 0.04 / 0.79 / 0.18)")

q_an = np.array([6, 0, 0])
weights_an = attention_weights(q_an, keys)
print(f"Writing 'an':   after unha={weights_an[0]:.2f}, after tía={weights_an[1]:.2f}, after miña={weights_an[2]:.2f}  (lecture: 0.75 / 0.17 / 0.08)")

q_mine = np.array([0, 0, 6])
weights_mine = attention_weights(q_mine, keys)
print(f"Writing 'mine': after unha={weights_mine[0]:.2f}, after tía={weights_mine[1]:.2f}, after miña={weights_mine[2]:.2f}  (lecture: 0.05 / 0.05 / 0.91)")


In [ ]:
# Check for Exercise 1. Run this cell as is.
assert np.allclose(softmax(np.array([0.0, 0.0])), [0.5, 0.5])
assert np.isclose(softmax(np.array([1000.0, 0.0]))[0], 1.0), "subtract the maximum first, or exp(1000) overflows"
assert np.allclose(weights_aunt, [0.0391, 0.7856, 0.1753], atol=1e-3), weights_aunt
assert np.allclose(weights_an, [0.7527, 0.1680, 0.0793], atol=1e-3), weights_an
assert np.allclose(weights_mine, [0.0453, 0.0453, 0.9094], atol=1e-3), weights_mine
print("✓ Exercise 1 passed")

<details><summary><b>Solution to Exercise 1</b> (open it after you have tried)</summary>

```python
import numpy as np

word_vectors = {"unha": np.array([1.0, 0, 0]), "tía": np.array([0, 1.0, 0]), "miña": np.array([0, 0, 1.0])}

encoder_vectors = []
v = np.zeros(3)
for word in ["unha", "tía", "miña"]:
    v = 0.5 * v + 0.5 * word_vectors[word]
    encoder_vectors.append(v)
keys = np.array(encoder_vectors)
print("keys:", keys.tolist())


def softmax(scores):
    exp_scores = np.exp(scores - np.max(scores))
    return exp_scores / exp_scores.sum()


def attention_weights(query, keys):
    scores = keys @ query
    return softmax(scores)


q_aunt = np.array([0, 6, 0])
weights_aunt = attention_weights(q_aunt, keys)
print(f"Writing 'aunt': after unha={weights_aunt[0]:.2f}, after tía={weights_aunt[1]:.2f}, after miña={weights_aunt[2]:.2f}  (lecture: 0.04 / 0.79 / 0.18)")

q_an = np.array([6, 0, 0])
weights_an = attention_weights(q_an, keys)
print(f"Writing 'an':   after unha={weights_an[0]:.2f}, after tía={weights_an[1]:.2f}, after miña={weights_an[2]:.2f}  (lecture: 0.75 / 0.17 / 0.08)")

q_mine = np.array([0, 0, 6])
weights_mine = attention_weights(q_mine, keys)
print(f"Writing 'mine': after unha={weights_mine[0]:.2f}, after tía={weights_mine[1]:.2f}, after miña={weights_mine[2]:.2f}  (lecture: 0.05 / 0.05 / 0.91)")
```

</details>

**Reflection:** These keys overlap: the key after *tía* also holds some *unha*, and the key after *miña* holds some of both. How does that change the weights? Try using the word vectors themselves as keys, `keys = np.array(list(word_vectors.values()))`, and rerun: what happens to the weight on *tía* for “aunt”, and why?


## Part B — Real attention weights, on our own sentence

The toy example above used hand-picked numbers. Here we load the real dedicated Galician→English model from Notebook 1 (`Helsinki-NLP/opus-mt-gl-en`), translate the exact fragment from lecture, and pull out its actual cross-attention weights instead of setting them by hand.


In [ ]:
!pip install -q transformers sentencepiece

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MT_MODEL_NAME = "Helsinki-NLP/opus-mt-gl-en"

mt_tokenizer = AutoTokenizer.from_pretrained(MT_MODEL_NAME)
mt_model = AutoModelForSeq2SeqLM.from_pretrained(MT_MODEL_NAME, attn_implementation="eager")
mt_model.eval()
print(f"Loaded {MT_MODEL_NAME}")


### ✏️ Exercise 2 — Average the real attention weights

`cross_attn` has shape **(layers, batch, heads, target tokens, source tokens)**.
Average it over layers and heads, and keep the first (and only) sentence of the batch, so
that `avg_attn` has one row per generated token and one column per source token.

<details><summary><b>Hints</b> (click to open)</summary>

- `tensor.mean(dim=(a, b))` averages over dimensions `a` and `b` and removes them.
- Dimensions are numbered from 0: layers is 0, batch is 1, heads is 2.

</details>

In [ ]:
FRAGMENT_GL = "unha tía miña"

inputs = mt_tokenizer(FRAGMENT_GL, return_tensors="pt")
with torch.no_grad():
    generated_ids = mt_model.generate(**inputs, max_new_tokens=20)

translation = mt_tokenizer.decode(generated_ids[0], skip_special_tokens=True)
print(f"Translation: \"{translation}\"")

with torch.no_grad():
    outputs = mt_model(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        decoder_input_ids=generated_ids,
        output_attentions=True,
        return_dict=True,
    )

cross_attn = torch.stack(outputs.cross_attentions)
print("cross_attn shape:", tuple(cross_attn.shape))
avg_attn = cross_attn.mean(dim=(___, ___))[0]

target_tokens = mt_tokenizer.convert_ids_to_tokens(generated_ids[0][1:])
avg_attn = avg_attn[:-1]

source_tokens = mt_tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

print(f"Source tokens: {source_tokens}")
print(f"Target tokens: {target_tokens}")


In [ ]:
# Check for Exercise 2. Run this cell as is.
assert tuple(avg_attn.shape) == (len(target_tokens), len(source_tokens)), tuple(avg_attn.shape)
assert torch.allclose(avg_attn.sum(dim=-1), torch.ones(len(target_tokens)), atol=1e-4), \
    "each row is an average of softmax rows, so it should sum to 1"
print("✓ Exercise 2 passed")

<details><summary><b>Solution to Exercise 2</b> (open it after you have tried)</summary>

```python
FRAGMENT_GL = "unha tía miña"

inputs = mt_tokenizer(FRAGMENT_GL, return_tensors="pt")
with torch.no_grad():
    generated_ids = mt_model.generate(**inputs, max_new_tokens=20)

translation = mt_tokenizer.decode(generated_ids[0], skip_special_tokens=True)
print(f"Translation: \"{translation}\"")

with torch.no_grad():
    outputs = mt_model(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        decoder_input_ids=generated_ids,
        output_attentions=True,
        return_dict=True,
    )

cross_attn = torch.stack(outputs.cross_attentions)
avg_attn = cross_attn.mean(dim=(0, 2))[0]

target_tokens = mt_tokenizer.convert_ids_to_tokens(generated_ids[0][1:])
avg_attn = avg_attn[:-1]

source_tokens = mt_tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

print(f"Source tokens: {source_tokens}")
print(f"Target tokens: {target_tokens}")
```

</details>

### ✏️ Exercise 3 — Read the alignment off the matrix

After the heatmap, list for each generated English token the Galician source token
it attends to most. Store the pairs in `alignment`.

<details><summary><b>Hints</b> (click to open)</summary>

- Row `i` of `avg_attn` holds the weights for generated token `i`.
- `row.argmax()` is the column with the largest weight.

</details>

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4))
im = ax.imshow(avg_attn.numpy(), cmap="Blues", aspect="auto")

ax.set_xticks(range(len(source_tokens)))
ax.set_xticklabels(source_tokens, rotation=45, ha="right")
ax.set_yticks(range(len(target_tokens)))
ax.set_yticklabels(target_tokens)
ax.set_xlabel("source (Galician)")
ax.set_ylabel("generated (English)")
ax.set_title("Real cross-attention weights, averaged over heads and layers")

fig.colorbar(im, ax=ax, label="attention weight")
plt.tight_layout()
plt.show()

alignment = []
for tgt_token, row in zip(target_tokens, avg_attn):
    best_src = source_tokens[int(row.___())]  # index of the largest weight in the row
    alignment.append((tgt_token, ___))
    print(f"{tgt_token:>12s} -> {best_src}")


In [ ]:
# Check for Exercise 3. Run this cell as is.
assert len(alignment) == len(target_tokens)
assert all(src == source_tokens[int(avg_attn[i].argmax())] for i, (_, src) in enumerate(alignment))
print("✓ Exercise 3 passed")

<details><summary><b>Solution to Exercise 3</b> (open it after you have tried)</summary>

```python
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4))
im = ax.imshow(avg_attn.numpy(), cmap="Blues", aspect="auto")

ax.set_xticks(range(len(source_tokens)))
ax.set_xticklabels(source_tokens, rotation=45, ha="right")
ax.set_yticks(range(len(target_tokens)))
ax.set_yticklabels(target_tokens)
ax.set_xlabel("source (Galician)")
ax.set_ylabel("generated (English)")
ax.set_title("Real cross-attention weights, averaged over heads and layers")

fig.colorbar(im, ax=ax, label="attention weight")
plt.tight_layout()
plt.show()

alignment = []
for tgt_token, row in zip(target_tokens, avg_attn):
    best_src = source_tokens[int(row.argmax())]
    alignment.append((tgt_token, best_src))
    print(f"{tgt_token:>12s} -> {best_src}")
```

</details>

**Reflection:** Compare the heatmap above to the toy example (79% on the key after *tía* for “aunt”). Is the real model's attention as concentrated on a single source token, or is it spread across more than one? Does any of the attention weight land somewhere that surprises you (e.g. on punctuation or the end-of-sequence token)? What does that tell you about the difference between the toy example from lecture and how attention actually behaves in a trained model?


## Code literacy check

Before you move on, pick at least two code cells above that you actually ran and read
today, and add `#` comments to them explaining, in your own words, what each meaningful
line or block does and *why* it's there, in language a classmate who hasn't seen the code
could follow.

Do this part on your own, after class:

1. Paste your commented code into an LLM chat (ChatGPT, Claude, Gemini — whichever you
   have access to).
2. Ask it something like: *"Here is code I commented myself, explaining what I think it
   does. Based only on my comments, how well do you think I understood this code? What
   did I get right, what did I miss, and what's still fuzzy?"*
3. Read what it says. Where it disagrees with your comment, go check the code again —
   who's right?

This is a self-check on your own understanding: a way to catch, in your own words, where
your mental model of the code breaks down against a second reader.
